In [1]:
import ModelCatalog
from utils import model_utils
from dataset.DataLoader import create_dataloaders
from FineTuner import fine_tune
from FineTuneStrategy import FineTuneStrategy

C:\Users\sinat\anaconda3\envs\breast-cancer-agent\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\sinat\anaconda3\envs\breast-cancer-agent\lib\site-packages\torch\utils\cpp_extension.py:28: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging  # type: ignore[attr-defined]


## Default model catalog
For the default catalog I have provided the agent with 5 models. 2 of these are vision transformers and the rest are classical deep convolutional nets.

In [2]:
catalog = ModelCatalog.get_default_catalog()
print(catalog)

Models:
Model name: vit_b_16, accuracy: 81.072, params: 86.6, gflops: 17.56
Model name: swin_s, accuracy: 83.196, params: 49.6, gflops: 8.74
Model name: resnet50, accuracy: 76.13, params: 25.6, gflops: 4.09
Model name: mobilenet_v3_large, accuracy: 74.042, params: 5.5, gflops: 0.22
Model name: densenet161, accuracy: 77.138, params: 28.7, gflops: 7.73



As you can see in the above cell, the param count ranges from 5.5M to 86.6M. The goal was to provide the model with different tools, each capable of achieving differing levels of accuracy with different parameter counts.

In [3]:
models_data = [
    catalog.get_model_data("vit_b_16"),
    catalog.get_model_data("swin_s"),
    catalog.get_model_data("resnet50"),
    catalog.get_model_data("densenet161"),
    catalog.get_model_data("mobilenet_v3_large")
]
models = [model_utils.get_model(model) for model in models_data]
print(models[0])

BreastCancerModel(
  (base_model): VisionTransformer(
    (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (encoder): Encoder(
      (dropout): Dropout(p=0.0, inplace=False)
      (layers): Sequential(
        (encoder_layer_0): EncoderBlock(
          (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (self_attention): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (dropout): Dropout(p=0.0, inplace=False)
          (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): MLPBlock(
            (0): Linear(in_features=768, out_features=3072, bias=True)
            (1): GELU(approximate='none')
            (2): Dropout(p=0.0, inplace=False)
            (3): Linear(in_features=3072, out_features=768, bias=True)
            (4): Dropout(p=0.0, inplace=False)
          )
        )
        (encoder_layer_1): EncoderBlock(
          

In [4]:
train_loader, val_loader, test_loader = create_dataloaders(batch_size=8)

Created Dataframes (800, 2), (100, 2), (100, 2)


In [ ]:
vit_lora = fine_tune(models[0],
                FineTuneStrategy.LORA,
                train_loader=train_loader,
                val_loader=val_loader,
                num_epochs=15,
                )
vit_lora

trainable params: 1,769,472 || all params: 87,571,973 || trainable%: 2.0206
class_weights: tensor([0.1548, 0.4595, 1.3293, 1.2407, 1.8157], device='cuda:0')

Starting fine-tuning with strategy: LORA
Device: cuda, Epochs: 15, LR: 0.0001
--------------------------------------------------------------------------------
Epoch 1/15
  Train Loss: 1.6585
  Val Loss: 1.5459
  Val Accuracy: 0.5000
  Val F1 (Weighted): 0.5153
  Val F1 (Macro): 0.3092
  Val Precision: 0.3318
  Val Recall: 0.3362
--------------------------------------------------------------------------------


In [8]:
resnet_full_ft = fine_tune(models[2],
                          strategy=FineTuneStrategy.FULL_FT,
                          train_loader=train_loader,
                          val_loader=val_loader)
resnet_full_ft

class_weights: tensor([0.1548, 0.4595, 1.3293, 1.2407, 1.8157], device='cuda:0')

Starting fine-tuning with strategy: FULL_FT
Device: cuda, Epochs: 10, LR: 0.0001
--------------------------------------------------------------------------------
Epoch 1/10
  Train Loss: 0.2743
  Val Loss: 3.0503
  Val Accuracy: 0.5300
  Val F1 (Weighted): 0.5202
  Val F1 (Macro): 0.2678
  Val Precision: 0.2484
  Val Recall: 0.3042
--------------------------------------------------------------------------------
Epoch 2/10
  Train Loss: 0.4244
  Val Loss: 2.7383
  Val Accuracy: 0.2800
  Val F1 (Weighted): 0.2957
  Val F1 (Macro): 0.1335
  Val Precision: 0.1565
  Val Recall: 0.1579
--------------------------------------------------------------------------------
Epoch 3/10
  Train Loss: 0.2645
  Val Loss: 3.4440
  Val Accuracy: 0.5100
  Val F1 (Weighted): 0.5034
  Val F1 (Macro): 0.2527
  Val Precision: 0.2573
  Val Recall: 0.2510
------------------------------------------------------------------------------

{'train_losses': [0.2742557725030929,
  0.42442012201994656,
  0.26454223349690437,
  0.17210358714219182,
  0.0944907254492864,
  0.044562329122563825,
  0.04786688786000013,
  0.02740443656919524,
  0.028652104319771752,
  0.016093024890869854],
 'val_losses': [3.050291643692897,
  2.738280612688798,
  3.443995326757431,
  3.5260234360511484,
  3.8674193001710453,
  4.073230688388531,
  3.919382236324824,
  3.8431562070663157,
  3.832034358038352,
  3.8834694486397963],
 'val_accuracies': [0.53,
  0.28,
  0.51,
  0.45,
  0.57,
  0.57,
  0.59,
  0.57,
  0.56,
  0.54],
 'val_f1_scores': [0.53, 0.28, 0.51, 0.45, 0.57, 0.57, 0.59, 0.57, 0.56, 0.54],
 'val_precision_scores': [0.2483695652173913,
  0.1564705882352941,
  0.2573076923076923,
  0.17261904761904762,
  0.1738562091503268,
  0.3931840311587147,
  0.3628787878787879,
  0.2801204819277109,
  0.23389450056116723,
  0.2576853526220615],
 'val_recall_scores': [0.30416666666666664,
  0.15788690476190476,
  0.25104166666666666,
  0.172

In [7]:
mobilenet_probe = fine_tune(models[4],
                            strategy=FineTuneStrategy.LINEAR_PROBE,
                            train_loader=train_loader,
                            val_loader=val_loader)
mobilenet_probe

class_weights: tensor([0.1548, 0.4595, 1.3293, 1.2407, 1.8157], device='cuda:0')

Starting fine-tuning with strategy: LINEAR_PROBE
Device: cuda, Epochs: 10, LR: 0.0001
--------------------------------------------------------------------------------
Epoch 1/10
  Train Loss: 1.6042
  Val Loss: 1.6893
  Val Accuracy: 0.2000
  Val F1 (Weighted): 0.2369
  Val F1 (Macro): 0.1365
  Val Precision: 0.1861
  Val Recall: 0.3192
--------------------------------------------------------------------------------
Epoch 2/10
  Train Loss: 1.5668
  Val Loss: 1.6983
  Val Accuracy: 0.2700
  Val F1 (Weighted): 0.3227
  Val F1 (Macro): 0.1652
  Val Precision: 0.1942
  Val Recall: 0.2814
--------------------------------------------------------------------------------
Epoch 3/10
  Train Loss: 1.5316
  Val Loss: 1.7014
  Val Accuracy: 0.2100
  Val F1 (Weighted): 0.2767
  Val F1 (Macro): 0.1441
  Val Precision: 0.1802
  Val Recall: 0.1594
-------------------------------------------------------------------------

{'train_losses': [1.6042066931724548,
  1.5668411087989806,
  1.5316148257255555,
  1.490725907087326,
  1.4860469853878022,
  1.4445349115133286,
  1.4415013813972473,
  1.4325019133090973,
  1.4250265860557556,
  1.429302569627762],
 'val_losses': [1.6892768878203173,
  1.6982845618174627,
  1.7013674424244807,
  1.7124478633587177,
  1.6822407245635986,
  1.656537257708036,
  1.6238117859913752,
  1.6006022416628325,
  1.5937040218940148,
  1.590677600640517],
 'val_accuracies': [0.2, 0.27, 0.21, 0.19, 0.2, 0.23, 0.37, 0.41, 0.43, 0.45],
 'val_f1_scores': [0.2, 0.27, 0.21, 0.19, 0.2, 0.23, 0.37, 0.41, 0.43, 0.45],
 'val_precision_scores': [0.1860640732265446,
  0.19420634920634922,
  0.18015873015873016,
  0.1706959706959707,
  0.1472972972972973,
  0.15151515151515152,
  0.1715170278637771,
  0.16416666666666666,
  0.17903225806451611,
  0.18787878787878787],
 'val_recall_scores': [0.31919642857142855,
  0.2813988095238095,
  0.159375,
  0.25,
  0.07529761904761904,
  0.09747023809